# SatyaScore — GPU runbook (Colab T4)

Runs the model-bound work (latency benchmark, retrieval diagnostic, and — behind a flag —
signal recompute + result regeneration) on a GPU and pushes generated results back to the repo.

**Every cell is re-runnable from scratch** (Colab sessions drop without warning). Only manual steps:
paste the GitHub token in Cell 2, and flip the two flags in Cells 8–9 if you want the heavy steps.

## Cell 1 — GPU guard
Stop immediately if no GPU was allocated; every downstream number is meaningless on CPU.

In [ ]:
!nvidia-smi
import torch
assert torch.cuda.is_available(), (
    "No GPU allocated. Runtime -> Change runtime type -> T4 GPU, then re-run this cell."
)
print("CUDA OK:", torch.cuda.get_device_name(0))

## Cell 2 — Clone or update the (private) repo
Token read via getpass — never hardcoded, never printed. Auth uses a per-command `git -c http.extraheader`, so the token never enters a URL, `.git/config`, or saved cell output; the stored remote stays token-free. Defines `git_push()` (return-code checked, output redacted) for the push cells below.

In [ ]:
import os, base64, getpass, subprocess
from pathlib import Path

GH_TOKEN = getpass.getpass("GitHub token (input hidden): ")  # kept in memory only
REPO_URL = "https://github.com/ViveekB410/SatyaScore.git"     # no token, ever

# Authenticate via a per-command header, NOT a token-in-URL. A token in the push
# URL broke curl with "Malformed input to a URL function"; it can also leak into
# .git/config or a saved cell output. `-c http.extraheader` is passed per command,
# is never written to config, and the stored remote stays clean.
_AUTH_B64 = base64.b64encode(("x-access-token:" + GH_TOKEN).encode()).decode()
AUTH_HDR = "AUTHORIZATION: basic " + _AUTH_B64


def _redact(s: str) -> str:
    for secret in (GH_TOKEN, _AUTH_B64):
        if secret:
            s = s.replace(secret, "***")
    return s


def git_auth(args):
    """Run `git -c http.extraheader=<auth> <args>`; returns the CompletedProcess."""
    return subprocess.run(["git", "-c", "http.extraheader=" + AUTH_HDR, *args],
                          capture_output=True, text=True)


def git_push(branch="main"):
    """Push HEAD to origin over the auth header. Checks the return code and prints
    a REDACTED result -- no more silent 'push attempted'. Returns True on success."""
    r = git_auth(["push", REPO_URL, "HEAD:" + branch])
    out = _redact((r.stdout or "") + (r.stderr or "")).strip()
    if r.returncode == 0:
        print("PUSH OK\n" + out)
        return True
    print("PUSH FAILED (exit %d):\n%s" % (r.returncode, out))
    return False


# clone or update using the header (never a token URL)
if Path("/content/SatyaScore/.git").exists():
    os.chdir("/content/SatyaScore")
    r = git_auth(["pull", REPO_URL, "main"])
else:
    r = git_auth(["clone", REPO_URL, "/content/SatyaScore"])
    os.chdir("/content/SatyaScore")
print(_redact((r.stdout or "") + (r.stderr or "")).strip()[-500:])

subprocess.run(["git", "remote", "set-url", "origin", REPO_URL])          # clean, token-free remote
subprocess.run(["git", "config", "user.email", "colab@satyascore.local"])  # identity for commits
subprocess.run(["git", "config", "user.name", "SatyaScore Colab"])
print("repo ready at", os.getcwd())

## Cell 3 — Install
Colab's CUDA torch is already correct (`requirements.txt` only asks `>=2.0.0`), so this won't reinstall it. `WANDB_MODE=disabled` stops wandb prompting.

In [ ]:
import os
os.environ["WANDB_MODE"] = "disabled"
!pip install -q -r requirements.txt
!python -m spacy download en_core_web_sm -q
print("install done")

## Cell 4 — Confirm the device
Both must print `cuda`. This is the checkpoint that catches a silent CPU fallback before any hours are spent.

In [ ]:
from member1_model.nli_scorer import NLIScorer
from member1_model.embeddings import CrossLingualEmbedder
print("NLI device     :", NLIScorer().device)
print("Embedder device:", CrossLingualEmbedder().device)

## Cell 5 — Test sanity
Expect ~114 passing (108 baseline + batching/span tests + 2 C2-lite + 2 NLI-chunking tests). If tests fail here, the environment is wrong — don't trust later numbers.

In [ ]:
!python -m pytest -q

## Cell 6 — Task A: latency benchmark
Reads the per-stage `timings_ms`; states whether the PRD targets are met on this GPU. Writes `results/latency_benchmark.md`.

In [ ]:
import os
from IPython.display import Markdown, display
if os.path.exists("benchmark_latency.py"):
    !python benchmark_latency.py
    if os.path.exists("results/latency_benchmark.md"):
        display(Markdown(open("results/latency_benchmark.md", encoding="utf-8").read()))
else:
    print("benchmark_latency.py not found — see PHASE3_IMPROVEMENTS.md Task A")

## Cell 7 — Task B: retrieval diagnostic
**This decides the direction of the fixes.** It states explicitly whether the evidence points to a retrieval (C1), granularity (C2), or model/label (C3) problem. Writes `results/retrieval_diagnostic.{md,csv}`.

In [ ]:
import os
from IPython.display import Markdown, display
if os.path.exists("diagnose_retrieval.py"):
    !python diagnose_retrieval.py
    if os.path.exists("results/retrieval_diagnostic.md"):
        display(Markdown(open("results/retrieval_diagnostic.md", encoding="utf-8").read()))
else:
    print("diagnose_retrieval.py not found — see PHASE3_IMPROVEMENTS.md Task B")

## Cell 7b — Task C2-lite: sentence-level NLI premises (A/B)
Runs `c2_lite_compare.py` with **no `--limit`** (authoritative — NLI is deterministic, so these numbers are final, not smoke). It scores NLI with whole-passage premises vs sentence-level premises on the same claims and reports **NLI AUC** (the deciding metric) and neutral rate, before and after, per language. The live API default stays **off** (`SATYASCORE_SENTENCE_PREMISES=false`) until this run shows it helps. Writes and commits `results/c2_lite_comparison.md`.

In [ ]:
import os, subprocess
from IPython.display import Markdown, display
if os.path.exists("c2_lite_compare.py"):
    !python c2_lite_compare.py            # full run, NO --limit (authoritative)
    p = "results/c2_lite_comparison.md"
    if os.path.exists(p):
        display(Markdown(open(p, encoding="utf-8").read()))
        # commit + push just this one result file (auth header from Cell 2)
        subprocess.run(["git", "add", p])
        c = subprocess.run(
            ["git", "commit", "-m",
             "Colab GPU: C2-lite comparison (sentence-level NLI premises, before/after AUC)"],
            capture_output=True, text=True)
        print((c.stdout or c.stderr).strip())
        if c.returncode == 0:
            git_push()                     # checks return code, prints redacted result
        else:
            print("nothing new committed — not pushing")
    else:
        print("c2_lite_compare.py ran but results/c2_lite_comparison.md was not produced")
else:
    print("c2_lite_compare.py not found — see TASK_C2_LITE.md")

## Cell 8 — Task D: recompute + regenerate (GUARDED)
Overwrites generated results. Flip `RUN_FULL_RECOMPUTE = True` **deliberately**, and only after reading Cell 7 and doing a Task C fix. Regenerates as a set (PHASE3 rule 3).

In [ ]:
RUN_FULL_RECOMPUTE = False  # <-- flip to True on purpose

if RUN_FULL_RECOMPUTE:
    !python recompute_signals.py --limit 6      # smoke test first
    !python recompute_signals.py                # full (writes gold_final_scored.csv)
    !python make_clean_split.py                 # -> gold_train_clean.csv / gold_test_clean.csv
    !python -m member1_model.train_weights --csv gold_train_clean.csv --languages en hi
    !python honest_ablation.py --train gold_train_clean.csv --test gold_test_clean.csv --out results/ablation_clean_no_semsim.json --table ablation_clean_no_semsim.md
    !python generate_results_summary.py         # -> RESULTS_SUMMARY.md
    print("recompute + regeneration complete")
else:
    print("RUN_FULL_RECOMPUTE is False — skipping (flip to True to regenerate everything).")

## Cell 9 — Optional: rebuild the bigger demo corpus (GUARDED)
GPU makes a larger demo corpus practical. Affects the live API demo only, never the evaluation.

In [ ]:
REBUILD_DEMO_CORPUS = False  # <-- flip to True on purpose

if REBUILD_DEMO_CORPUS:
    !python build_demo_corpus.py
    print("demo corpus rebuilt")
else:
    print("REBUILD_DEMO_CORPUS is False — skipping.")

## Cell 10 — Commit + push generated results
Stages only generated outputs that exist (never `.venv`, caches, or checkpoints). Pushes via `git_push()` from Cell 2 (auth header, never a token URL); checks the return code and prints a redacted result instead of a blind "push attempted".

In [ ]:
import subprocess, os
# stage only the generated outputs that actually exist (avoids a git-add error)
paths = [x for x in ["results", "Figures", "RESULTS_SUMMARY.md", "HANDOFF.md",
                     "ablation_clean_no_semsim.md"] if os.path.exists(x)]
subprocess.run(["git", "add", *paths])
subprocess.run(["git", "status", "--short"])
c = subprocess.run(
    ["git", "commit", "-m", "Colab GPU: regenerated results (benchmark / diagnostic / eval)"],
    capture_output=True, text=True)
print((c.stdout or c.stderr).strip())
if c.returncode == 0:
    git_push()                     # auth header from Cell 2; checks return code
else:
    print("nothing new committed — not pushing")